In [1]:
import os

from io import StringIO
from google.cloud import storage
from dotenv import load_dotenv

import pandas as pd
import numpy as np
import matplotlib.pyplot as plt

import torch
import torch.nn.functional as F

from transformers import AutoModelForCausalLM
from transformers import AutoTokenizer, EsmForMaskedLM
from tokenizers import Tokenizer
from peft import get_peft_model, LoraConfig, TaskType


/opt/anaconda3/envs/MachLearn/lib/python3.12/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [2]:
from scipy.stats import ttest_rel

In [3]:
from plm_compare_progen2 import *
from plm_compare_esm import *
from protein_data import *
from pro_gen2_lora import *

In [ ]:
# with open('/Users/johnhutchens/Desktop/Practicum/Data/Wild_Dictionaries/pg2_ProGym_matrices.pickle',
#            'rb') as f:
#     pg_dict = pickle.load(f)

In [4]:
path = '/Users/johnhutchens/Desktop/Practicum/Data/Domainome/'

with open(path+"dict_PF00030.pkl", "rb") as f:
    dict_PF00030 = pickle.load(f)

In [5]:
dict_PF00030

{'P05813_PF00030_31': {'DMS_mat': array([[-0.10443451,  0.13707808,  0.02807782, ..., -0.03048719,
                  nan,  0.37133969],
         [-0.47260158, -0.33754298, -1.00977863, ..., -0.37827595,
          -0.68666695, -1.01045355],
         [        nan, -0.46455143,         nan, ..., -0.53509969,
          -0.81186809,         nan],
         ...,
         [ 0.01572358,         nan,  0.13154921, ...,  0.00885801,
          -0.10067819,  0.00898411],
         [ 0.12593207, -0.19203179,  0.00742594, ..., -0.22169962,
          -0.51913589, -0.11086738],
         [        nan, -0.23419564, -0.15285446, ..., -0.14229641,
          -0.19328258, -0.29164243]], shape=(89, 20)),
  'wt_seq': 'WKITIYDQENFQGKRMEFTSSCPNVSERSFDNVRSLKVESGAWIGYEHTSFCGQQFILERGEYPRWDAWSGSNAYHIERLMSFRPICSA',
  'llr_pg2_lora_eps30_lr1eneg3_PF00030': array([[ 0.2697296 , -0.10356903, -0.16577148, ...,  0.05006409,
           0.        , -0.05153656],
         [ 0.81728363,  0.20957947,  0.28593445, ...,  0.6034393

In [5]:
keys = list(dict_PF00030.keys())

In [7]:
for key in keys:
    print(len(dict_PF00030[key]['DMS_mat']))

89
76
80
84
87
81
89
81
84
83
87
78


In [6]:
keys = list(dict_PF00030.keys())

Create testing data from PF00030 to test model trained on P07316_PF00030_87

In [4]:
# load_dotenv()
# cred_path = os.getenv('GOOGLE_APPLICATION_CREDENTIALS')

# os.environ['GOOGLE_APPLICATION_CREDENTIALS'] = cred_path

# client = storage.Client()
# bucket = client.bucket('domainome-data')
# blob = bucket.blob('SupplementaryTable2.txt')

# df = pd.read_csv(StringIO(blob.download_as_text()), sep='\t')

In [ ]:
# df.head()

,domain_ID,uniprot_ID,aa_seq,wt_aa,position,mut_aa,STOP,input_count_rep1,input_count_rep2,input_count_rep3,output_count_rep1,output_count_rep2,output_count_rep3,mean_input_count,fitness,fitness_sigma,normalized_fitness,normalized_fitness_sigma,quality_rank
0,A0A2R8Y422_PF00240_2,A0A2R8Y422,*IFVKTLMGKTITLEVELSDTIDNVKAKIQDKEGIPPDQQRLIFAG...,Q,2.0,*,True,118.0,113.0,62.0,10.0,29.0,2.0,97.66667,0.030945,0.014885,-0.819050,0.208478,339
1,A0A2R8Y422_PF00240_2,A0A2R8Y422,AIFVKTLMGKTITLEVELSDTIDNVKAKIQDKEGIPPDQQRLIFAG...,Q,2.0,A,False,219.0,277.0,217.0,86.0,225.0,137.0,237.66670,0.069376,0.006673,-0.280790,0.093461,339
2,A0A2R8Y422_PF00240_2,A0A2R8Y422,CIFVKTLMGKTITLEVELSDTIDNVKAKIQDKEGIPPDQQRLIFAG...,Q,2.0,C,False,706.0,726.0,459.0,768.0,507.0,616.0,630.33330,0.082052,0.004141,-0.103250,0.057995,339
3,A0A2R8Y422_PF00240_2,A0A2R8Y422,DIFVKTLMGKTITLEVELSDTIDNVKAKIQDKEGIPPDQQRLIFAG...,Q,2.0,D,False,407.0,431.0,323.0,508.0,159.0,111.0,387.00000,0.071003,0.005162,-0.258003,0.072296,339
4,A0A2R8Y422_PF00240_2,A0A2R8Y422,EIFVKTLMGKTITLEVELSDTIDNVKAKIQDKEGIPPDQQRLIFAG...,Q,2.0,E,False,37.0,56.0,37.0,201.0,102.0,95.0,43.33333,0.116326,0.012085,0.376783,0.169263,339


Load base model

In [7]:
device = 'cpu'
print(f"Using {device} device")
model_name = "hugohrban/progen2-medium"
base_model, tokenizer = initialize_progen2_noeval(model_name)

Using cpu device


# Changing the num_samples, train/val only

# Fine tune model using 5 epochs, learning rate 1e-3, CausalLM, num_samples = 10

In [15]:
lora_config = LoraConfig(
    r=8,
    lora_alpha=32,
    target_modules=["qkv_proj", "out_proj"],
    lora_dropout=0.1,
    bias="none",
    # task_type=TaskType.FEATURE_EXTRACTION
    task_type=TaskType.CAUSAL_LM
)

# protein_seq = 'EDGINLEEIREFAKNFKIRRLSLGLTQTQVGQALTATEGPAYSQSAICRFEKLDITPKSAQKLKPVLEKWLNEAELRNQEGQQNLMEFVG'
key = keys[0]
protein_seq = dict_PF00030[key]['wt_seq']

df_mutation = pd.read_csv('mutation.csv') # automate df_mutation
fitness_data = df_mutation
fitness_data.reset_index(drop=True, inplace=True)
positions = np.arange(len(protein_seq))

amino_acids = 'ACDEFGHIKLMNPQRSTVWY'
aa_token_ids = [tokenizer.convert_tokens_to_ids(aa) for aa in amino_acids]
positions_col = np.repeat(positions, len(amino_acids))
amino_acids_col = np.tile(list(amino_acids), len(positions))
df2 = pd.DataFrame({'real_position': positions_col, 'mut_aa': amino_acids_col})
df_merged = df2.merge(
    fitness_data,
    on=['real_position', 'mut_aa'],
    how='left')
fitness_list = df_merged['normalized_fitness'].tolist()
fitness_tensor = torch.tensor(fitness_list)
exp_tensor = fitness_tensor
seq_len = exp_tensor.shape[0]

loss = listwise_ranking_loss


# num_epochs = [30]
# l_rates = [1e-3, 5e-3]
# len_ep = len(num_epochs)
# len_lr = len(l_rates)

# results = {}


# for i in range(len_ep):
#     eps = num_epochs[i]
#     results[eps] = []
#     # print(f"Number of epochs: {eps}")
#     for j in range(len_lr):
#         lr = l_rates[j]
#         # print(f"Learning rate: {lr}")
eps = 5
lr = 1e-3
        
model, train_losses, val_losses = FineTune_ProGen2_LORA(device, base_model, tokenizer, 
                                        lora_config, protein_seq, exp_tensor, 
                                        loss, lrate=lr, num_epochs=eps, k=0.8, 
                                        num_samples=10, print_info=False)

print(f"Training loss = {train_losses}")
print(f"Validation loss = {val_losses}")
        

Training loss = [5.403512954711914, 2.5162763595581055, 2.037637233734131, 4.496270179748535, 2.0264930725097656]
Validation loss = [8.879287719726562, 2.8517394065856934, 2.635200023651123, 1.4528796672821045, 2.080329418182373]


In [16]:
model.eval()
ft_model_name = 'llr_pg2_lora_eps5_lr1eneg3_PF00030_CLM_ns10'

In [17]:
for key in dict_PF00030.keys():
    seq = dict_PF00030[key]['wt_seq']
    lp, rlp, llr = collect_log_prob_pg2(seq, model, tokenizer)
    dict_PF00030[key][ft_model_name] = llr

In [18]:
i = 0

dict_PF00030[keys[i]]['llr_pg2'] - dict_PF00030[keys[i]][ft_model_name]

array([[-1.7805862 , -1.1406479 , -2.1856842 , ..., -1.6869583 ,
         0.        , -0.45217133],
       [-1.0560759 , -0.72468543, -1.6307831 , ..., -0.98950195,
        -1.0013123 , -1.1443634 ],
       [-1.3891753 , -1.4149857 , -2.6120834 , ..., -0.54618824,
        -1.7802048 , -1.3778381 ],
       ...,
       [-3.6620865 ,  0.        , -3.0132828 , ..., -3.6633606 ,
        -5.1788177 , -2.684578  ],
       [-1.313736  ,  1.0758514 , -3.3888931 , ..., -2.6410828 ,
        -3.7852252 , -1.9599152 ],
       [ 0.        , -2.9024353 , -2.888611  , ..., -2.6646118 ,
        -5.107788  , -4.125412  ]], shape=(89, 20), dtype=float32)

In [19]:
SpearmanBase = []
SpearmanLoRA = []

for key in keys:
    base_llr = dict_PF00030[key]['llr_pg2']
    lora_llr = dict_PF00030[key][ft_model_name]
    dms_mat = dict_PF00030[key]['DMS_mat']

    # print(len(base_llr), len(lora_llr), len(dms_mat))


    spb = spearman_ignore_nan(base_llr, dms_mat)
    spl = spearman_ignore_nan(lora_llr, dms_mat)

    SpearmanBase.append(spb[0])
    SpearmanLoRA.append(spl[0])

In [20]:
for i in range(len(SpearmanBase)):
    spb = SpearmanBase[i]
    spl = SpearmanLoRA[i]
    print(f"Base: {spb}")
    print(f"LoRA: {spl}")
    print()

Base: 0.4172571483248994
LoRA: 0.46475694925858335

Base: 0.33897700677632076
LoRA: 0.2995121286851658

Base: 0.36205722660833856
LoRA: 0.29460284139094595

Base: 0.21845045525531706
LoRA: 0.17477531685088293

Base: 0.3662332108928841
LoRA: 0.30324808641318557

Base: 0.2790765656023974
LoRA: 0.24093716728653675

Base: 0.43572135931376255
LoRA: 0.369694564719311

Base: 0.31101513899232713
LoRA: 0.27238638077697774

Base: 0.26475198062516286
LoRA: 0.261736694527714

Base: 0.3800703713136749
LoRA: 0.3240140290615223

Base: 0.184950854040034
LoRA: 0.24604181075794887

Base: 0.35140039018183716
LoRA: 0.27221391311919413



In [21]:
t_stat, p_value = ttest_rel(SpearmanBase,SpearmanLoRA)
print(p_value)

0.030781020358046095


# Fine tune model using 5 epochs, learning rate 1e-3, CausalLM, num_samples = 40

In [8]:
lora_config = LoraConfig(
    r=8,
    lora_alpha=32,
    target_modules=["qkv_proj", "out_proj"],
    lora_dropout=0.1,
    bias="none",
    # task_type=TaskType.FEATURE_EXTRACTION
    task_type=TaskType.CAUSAL_LM
)

# protein_seq = 'EDGINLEEIREFAKNFKIRRLSLGLTQTQVGQALTATEGPAYSQSAICRFEKLDITPKSAQKLKPVLEKWLNEAELRNQEGQQNLMEFVG'
key = keys[0]
protein_seq = dict_PF00030[key]['wt_seq']

df_mutation = pd.read_csv('mutation.csv') # automate df_mutation
fitness_data = df_mutation
fitness_data.reset_index(drop=True, inplace=True)
positions = np.arange(len(protein_seq))

amino_acids = 'ACDEFGHIKLMNPQRSTVWY'
aa_token_ids = [tokenizer.convert_tokens_to_ids(aa) for aa in amino_acids]
positions_col = np.repeat(positions, len(amino_acids))
amino_acids_col = np.tile(list(amino_acids), len(positions))
df2 = pd.DataFrame({'real_position': positions_col, 'mut_aa': amino_acids_col})
df_merged = df2.merge(
    fitness_data,
    on=['real_position', 'mut_aa'],
    how='left')
fitness_list = df_merged['normalized_fitness'].tolist()
fitness_tensor = torch.tensor(fitness_list)
exp_tensor = fitness_tensor
seq_len = exp_tensor.shape[0]

loss = listwise_ranking_loss


# num_epochs = [30]
# l_rates = [1e-3, 5e-3]
# len_ep = len(num_epochs)
# len_lr = len(l_rates)

# results = {}


# for i in range(len_ep):
#     eps = num_epochs[i]
#     results[eps] = []
#     # print(f"Number of epochs: {eps}")
#     for j in range(len_lr):
#         lr = l_rates[j]
#         # print(f"Learning rate: {lr}")
eps = 5
lr = 1e-3
        
model, train_losses, val_losses = FineTune_ProGen2_LORA(device, base_model, tokenizer, 
                                        lora_config, protein_seq, exp_tensor, 
                                        loss, lrate=lr, num_epochs=eps, k=0.8, 
                                        num_samples=40, print_info=False)

print(f"Training loss = {train_losses}")
print(f"Validation loss = {val_losses}")
        

Training loss = [6.5764336585998535, 5.786240100860596, 4.082671165466309, 3.0122482776641846, 3.1828436851501465]
Validation loss = [8.93297290802002, 5.237157344818115, 3.620352268218994, 3.261817216873169, 3.030447483062744]


In [9]:
model.eval()
ft_model_name = 'llr_pg2_lora_eps5_lr1eneg3_PF00030_CLM_ns40'

In [10]:
for key in dict_PF00030.keys():
    seq = dict_PF00030[key]['wt_seq']
    lp, rlp, llr = collect_log_prob_pg2(seq, model, tokenizer)
    dict_PF00030[key][ft_model_name] = llr

In [11]:
i = 0

dict_PF00030[keys[i]]['llr_pg2'] - dict_PF00030[keys[i]][ft_model_name]

array([[-1.4291077 , -0.96684265, -2.1694262 , ..., -1.6117249 ,
         0.        , -0.02457428],
       [-1.8168182 , -1.9846036 , -2.622757  , ..., -1.9904175 ,
        -3.1830292 , -2.8275375 ],
       [-1.9161148 , -2.5897064 , -3.4197083 , ..., -0.61306   ,
        -3.861374  , -2.679184  ],
       ...,
       [-3.1298904 ,  0.        , -2.7935715 , ..., -3.5385818 ,
        -6.38797   , -3.204544  ],
       [-1.4543152 ,  0.40469354, -3.8770905 , ..., -3.0529785 ,
        -5.666115  , -3.3366773 ],
       [ 0.        , -3.6659164 , -3.4661944 , ..., -3.1143646 ,
        -6.973686  , -5.496582  ]], shape=(89, 20), dtype=float32)

In [12]:
SpearmanBase = []
SpearmanLoRA = []

for key in keys:
    base_llr = dict_PF00030[key]['llr_pg2']
    lora_llr = dict_PF00030[key][ft_model_name]
    dms_mat = dict_PF00030[key]['DMS_mat']

    # print(len(base_llr), len(lora_llr), len(dms_mat))


    spb = spearman_ignore_nan(base_llr, dms_mat)
    spl = spearman_ignore_nan(lora_llr, dms_mat)

    SpearmanBase.append(spb[0])
    SpearmanLoRA.append(spl[0])

In [13]:
for i in range(len(SpearmanBase)):
    spb = SpearmanBase[i]
    spl = SpearmanLoRA[i]
    print(f"Base: {spb}")
    print(f"LoRA: {spl}")
    print()

Base: 0.4172571483248994
LoRA: 0.36802936431511024

Base: 0.33897700677632076
LoRA: 0.25334333093567807

Base: 0.36205722660833856
LoRA: 0.2132439388519887

Base: 0.21845045525531706
LoRA: 0.15894248196612654

Base: 0.3662332108928841
LoRA: 0.25052967579564217

Base: 0.2790765656023974
LoRA: 0.23502393362027466

Base: 0.43572135931376255
LoRA: 0.30488556129105854

Base: 0.31101513899232713
LoRA: 0.23625020629675084

Base: 0.26475198062516286
LoRA: 0.2574553976471904

Base: 0.3800703713136749
LoRA: 0.2651826947488685

Base: 0.184950854040034
LoRA: 0.22277523254981746

Base: 0.35140039018183716
LoRA: 0.20811710197122468



In [14]:
t_stat, p_value = ttest_rel(SpearmanBase,SpearmanLoRA)
print(p_value)

0.0006027711103973964


# Fine tune model using 5 epochs, learning rate 1e-3, CausalLM

In [7]:
lora_config = LoraConfig(
    r=8,
    lora_alpha=32,
    target_modules=["qkv_proj", "out_proj"],
    lora_dropout=0.1,
    bias="none",
    # task_type=TaskType.FEATURE_EXTRACTION
    task_type=TaskType.CAUSAL_LM
)

# protein_seq = 'EDGINLEEIREFAKNFKIRRLSLGLTQTQVGQALTATEGPAYSQSAICRFEKLDITPKSAQKLKPVLEKWLNEAELRNQEGQQNLMEFVG'
key = keys[0]
protein_seq = dict_PF00030[key]['wt_seq']

df_mutation = pd.read_csv('mutation.csv') # automate df_mutation
fitness_data = df_mutation
fitness_data.reset_index(drop=True, inplace=True)
positions = np.arange(len(protein_seq))

amino_acids = 'ACDEFGHIKLMNPQRSTVWY'
aa_token_ids = [tokenizer.convert_tokens_to_ids(aa) for aa in amino_acids]
positions_col = np.repeat(positions, len(amino_acids))
amino_acids_col = np.tile(list(amino_acids), len(positions))
df2 = pd.DataFrame({'real_position': positions_col, 'mut_aa': amino_acids_col})
df_merged = df2.merge(
    fitness_data,
    on=['real_position', 'mut_aa'],
    how='left')
fitness_list = df_merged['normalized_fitness'].tolist()
fitness_tensor = torch.tensor(fitness_list)
exp_tensor = fitness_tensor
seq_len = exp_tensor.shape[0]

loss = listwise_ranking_loss


# num_epochs = [30]
# l_rates = [1e-3, 5e-3]
# len_ep = len(num_epochs)
# len_lr = len(l_rates)

# results = {}


# for i in range(len_ep):
#     eps = num_epochs[i]
#     results[eps] = []
#     # print(f"Number of epochs: {eps}")
#     for j in range(len_lr):
#         lr = l_rates[j]
#         # print(f"Learning rate: {lr}")
eps = 5
lr = 1e-3
        
model, train_losses, val_losses = FineTune_ProGen2_LORA(device, base_model, tokenizer, 
                                        lora_config, protein_seq, exp_tensor, 
                                        loss, lrate=lr, num_epochs=eps, k=0.5, 
                                        num_samples=20, print_info=False)

print(f"Training loss = {train_losses}")
print(f"Validation loss = {val_losses}")
        

Training loss = [5.260550498962402, 2.5332024097442627, 3.372142791748047, 2.908649206161499, 1.829838752746582]
Validation loss = [3.0650134086608887, 6.4389142990112305, 3.0385022163391113, 2.2131755352020264, 2.498478412628174]


Computing LLR matrices using fine-tune model

In [8]:
model.eval()

PeftModelForCausalLM(
  (base_model): LoraModel(
    (model): ProGenForCausalLM(
      (transformer): ProGenModel(
        (wte): Embedding(32, 1536)
        (drop): Dropout(p=0.0, inplace=False)
        (h): ModuleList(
          (0-26): 27 x ProGenBlock(
            (ln_1): LayerNorm((1536,), eps=1e-05, elementwise_affine=True)
            (attn): ProGenAttention(
              (attn_dropout): Dropout(p=0.0, inplace=False)
              (resid_dropout): Dropout(p=0.0, inplace=False)
              (qkv_proj): lora.Linear(
                (base_layer): Linear(in_features=1536, out_features=4608, bias=False)
                (lora_dropout): ModuleDict(
                  (default): Dropout(p=0.1, inplace=False)
                )
                (lora_A): ModuleDict(
                  (default): Linear(in_features=1536, out_features=8, bias=False)
                )
                (lora_B): ModuleDict(
                  (default): Linear(in_features=8, out_features=4608, bias=False)
      

In [9]:
ft_model_name = 'llr_pg2_lora_eps5_lr1eneg3_PF00030_CLM'

In [10]:
for key in dict_PF00030.keys():
    seq = dict_PF00030[key]['wt_seq']
    lp, rlp, llr = collect_log_prob_pg2(seq, model, tokenizer)
    dict_PF00030[key][ft_model_name] = llr

In [11]:
device = 'cpu'
print(f"Using {device} device")
model_name = "hugohrban/progen2-medium"
base_model, tokenizer = initialize_progen2_noeval(model_name)

Using cpu device


In [12]:
base_model.eval()

ProGenForCausalLM(
  (transformer): ProGenModel(
    (wte): Embedding(32, 1536)
    (drop): Dropout(p=0.0, inplace=False)
    (h): ModuleList(
      (0-26): 27 x ProGenBlock(
        (ln_1): LayerNorm((1536,), eps=1e-05, elementwise_affine=True)
        (attn): ProGenAttention(
          (attn_dropout): Dropout(p=0.0, inplace=False)
          (resid_dropout): Dropout(p=0.0, inplace=False)
          (qkv_proj): Linear(in_features=1536, out_features=4608, bias=False)
          (out_proj): Linear(in_features=1536, out_features=1536, bias=False)
        )
        (mlp): ProGenMLP(
          (fc_in): Linear(in_features=1536, out_features=6144, bias=True)
          (fc_out): Linear(in_features=6144, out_features=1536, bias=True)
          (act): NewGELUActivation()
          (dropout): Dropout(p=0.0, inplace=False)
        )
      )
    )
    (ln_f): LayerNorm((1536,), eps=1e-05, elementwise_affine=True)
  )
  (lm_head): Linear(in_features=1536, out_features=32, bias=True)
)

In [14]:
i = 0

dict_PF00030[keys[i]]['llr_pg2'] - dict_PF00030[keys[i]][ft_model_name]

array([[-1.2624817 , -0.8487549 , -1.810982  , ..., -1.545517  ,
         0.        , -0.55914307],
       [-1.1810304 , -1.3902433 , -1.6130066 , ..., -1.4779054 ,
        -1.925705  , -1.9196777 ],
       [-1.0403748 , -1.797035  , -2.2122648 , ..., -0.33535004,
        -2.3332517 , -1.7674408 ],
       ...,
       [-2.6438522 ,  0.        , -2.9620438 , ..., -3.0952911 ,
        -5.3216705 , -3.180191  ],
       [-1.1033096 ,  0.5032043 , -3.7488785 , ..., -2.4773483 ,
        -4.601883  , -3.0935514 ],
       [ 0.        , -3.2847748 , -3.078438  , ..., -2.532837  ,
        -5.6741943 , -5.0941315 ]], shape=(89, 20), dtype=float32)

In [20]:
SpearmanBase = []
SpearmanLoRA = []

for key in keys:
    base_llr = dict_PF00030[key]['llr_pg2']
    lora_llr = dict_PF00030[key][ft_model_name]
    dms_mat = dict_PF00030[key]['DMS_mat']

    # print(len(base_llr), len(lora_llr), len(dms_mat))


    spb = spearman_ignore_nan(base_llr, dms_mat)
    spl = spearman_ignore_nan(lora_llr, dms_mat)

    SpearmanBase.append(spb[0])
    SpearmanLoRA.append(spl[0])


    # print(f"Base model spearman correlation with DMS is {spb[0]}")
    # print(f"Lora model spearman correlation with DMS is {spl[0]}")
    # print()

In [22]:
t_stat, p_value = ttest_rel(SpearmanBase,SpearmanLoRA)
print(p_value)

0.5795587054215662


# Fine tune model using 50 epochs, learning rate 1e-3, CausalLM

In [10]:
lora_config = LoraConfig(
    r=8,
    lora_alpha=32,
    target_modules=["qkv_proj", "out_proj"],
    lora_dropout=0.1,
    bias="none",
    # task_type=TaskType.FEATURE_EXTRACTION
    task_type=TaskType.CAUSAL_LM
)

# protein_seq = 'EDGINLEEIREFAKNFKIRRLSLGLTQTQVGQALTATEGPAYSQSAICRFEKLDITPKSAQKLKPVLEKWLNEAELRNQEGQQNLMEFVG'
key = keys[0]
protein_seq = dict_PF00030[key]['wt_seq']

df_mutation = pd.read_csv('mutation.csv') # automate df_mutation
fitness_data = df_mutation
fitness_data.reset_index(drop=True, inplace=True)
positions = np.arange(len(protein_seq))

amino_acids = 'ACDEFGHIKLMNPQRSTVWY'
aa_token_ids = [tokenizer.convert_tokens_to_ids(aa) for aa in amino_acids]
positions_col = np.repeat(positions, len(amino_acids))
amino_acids_col = np.tile(list(amino_acids), len(positions))
df2 = pd.DataFrame({'real_position': positions_col, 'mut_aa': amino_acids_col})
df_merged = df2.merge(
    fitness_data,
    on=['real_position', 'mut_aa'],
    how='left')
fitness_list = df_merged['normalized_fitness'].tolist()
fitness_tensor = torch.tensor(fitness_list)
exp_tensor = fitness_tensor
seq_len = exp_tensor.shape[0]

loss = listwise_ranking_loss


# num_epochs = [30]
# l_rates = [1e-3, 5e-3]
# len_ep = len(num_epochs)
# len_lr = len(l_rates)

# results = {}


# for i in range(len_ep):
#     eps = num_epochs[i]
#     results[eps] = []
#     # print(f"Number of epochs: {eps}")
#     for j in range(len_lr):
#         lr = l_rates[j]
#         # print(f"Learning rate: {lr}")
eps = 50
lr = 1e-3
        
model, val_loss = FineTune_ProGen2_LORA(device, base_model, tokenizer, 
                                        lora_config, protein_seq, exp_tensor, 
                                        loss, lrate=lr, num_epochs=eps, k=0.5, 
                                        num_samples=20, print_info=False)

print(f"Validation loss = {val_loss}")
        

Validation loss = 2.073259115219116


save fine-tuned model

In [ ]:
# # save a model
# model_dir = '/Users/johnhutchens/Desktop/Practicum/Models/'+'pg2_lora_eps50_lr1eneg3_PF00030_CLM'

# model.save_pretrained(model_dir)
# tokenizer.save_pretrained(model_dir)

('/Users/johnhutchens/Desktop/Practicum/Models/pg2_lora_eps50_lr1eneg3_PF00030_CLM/tokenizer_config.json',
 '/Users/johnhutchens/Desktop/Practicum/Models/pg2_lora_eps50_lr1eneg3_PF00030_CLM/special_tokens_map.json',
 '/Users/johnhutchens/Desktop/Practicum/Models/pg2_lora_eps50_lr1eneg3_PF00030_CLM/vocab.json',
 '/Users/johnhutchens/Desktop/Practicum/Models/pg2_lora_eps50_lr1eneg3_PF00030_CLM/merges.txt',
 '/Users/johnhutchens/Desktop/Practicum/Models/pg2_lora_eps50_lr1eneg3_PF00030_CLM/added_tokens.json',
 '/Users/johnhutchens/Desktop/Practicum/Models/pg2_lora_eps50_lr1eneg3_PF00030_CLM/tokenizer.json')

load fine-tuned model

In [ ]:
# model_dir = '/Users/johnhutchens/Desktop/Practicum/Models/'+'pg2_lora_eps30_lr1eneg3_PF00030'

# base_model = AutoModelForCausalLM.from_pretrained(base_model_name)
# model = PeftModel.from_pretrained(base_model, model_dir)
# tokenizer = AutoTokenizer.from_pretrained(model_dir)

construct LLR matrices using fine-tuned model

In [20]:
model.eval()

PeftModelForCausalLM(
  (base_model): LoraModel(
    (model): ProGenForCausalLM(
      (transformer): ProGenModel(
        (wte): Embedding(32, 1536)
        (drop): Dropout(p=0.0, inplace=False)
        (h): ModuleList(
          (0-26): 27 x ProGenBlock(
            (ln_1): LayerNorm((1536,), eps=1e-05, elementwise_affine=True)
            (attn): ProGenAttention(
              (attn_dropout): Dropout(p=0.0, inplace=False)
              (resid_dropout): Dropout(p=0.0, inplace=False)
              (qkv_proj): lora.Linear(
                (base_layer): Linear(in_features=1536, out_features=4608, bias=False)
                (lora_dropout): ModuleDict(
                  (default): Dropout(p=0.1, inplace=False)
                )
                (lora_A): ModuleDict(
                  (default): Linear(in_features=1536, out_features=8, bias=False)
                )
                (lora_B): ModuleDict(
                  (default): Linear(in_features=8, out_features=4608, bias=False)
      

In [21]:
for key in dict_PF00030.keys():
    seq = dict_PF00030[key]['wt_seq']
    lp, rlp, llr = collect_log_prob_pg2(seq, model, tokenizer)
    dict_PF00030[key]['llr_pg2_lora_eps50_lr1eneg3_PF00030_CLM'] = llr

reset base model (may not be necessary...)

In [22]:
device = 'cpu'
print(f"Using {device} device")
model_name = "hugohrban/progen2-medium"
base_model, tokenizer = initialize_progen2_noeval(model_name)

Using cpu device


construct LLR matrices using base model

In [23]:
base_model.eval()

ProGenForCausalLM(
  (transformer): ProGenModel(
    (wte): Embedding(32, 1536)
    (drop): Dropout(p=0.0, inplace=False)
    (h): ModuleList(
      (0-26): 27 x ProGenBlock(
        (ln_1): LayerNorm((1536,), eps=1e-05, elementwise_affine=True)
        (attn): ProGenAttention(
          (attn_dropout): Dropout(p=0.0, inplace=False)
          (resid_dropout): Dropout(p=0.0, inplace=False)
          (qkv_proj): Linear(in_features=1536, out_features=4608, bias=False)
          (out_proj): Linear(in_features=1536, out_features=1536, bias=False)
        )
        (mlp): ProGenMLP(
          (fc_in): Linear(in_features=1536, out_features=6144, bias=True)
          (fc_out): Linear(in_features=6144, out_features=1536, bias=True)
          (act): NewGELUActivation()
          (dropout): Dropout(p=0.0, inplace=False)
        )
      )
    )
    (ln_f): LayerNorm((1536,), eps=1e-05, elementwise_affine=True)
  )
  (lm_head): Linear(in_features=1536, out_features=32, bias=True)
)

In [24]:
for key in dict_PF00030.keys():
    seq = dict_PF00030[key]['wt_seq']
    lp, rlp, llr = collect_log_prob_pg2(seq, base_model, tokenizer)
    dict_PF00030[key]['llr_pg2'] = llr

quick check for difference in model outputs

In [42]:
i = 0

dict_PF00030[keys[i]]['llr_pg2'] - dict_PF00030[keys[i]]['llr_pg2_lora_eps50_lr1eneg3_PF00030_CLM']

array([[-3.5778732, -3.1686707, -3.9479983, ..., -3.6606445,  0.       ,
        -1.4397736],
       [-3.299385 , -3.742729 , -3.9519424, ..., -3.5338516, -3.8312607,
        -4.334999 ],
       [-2.810051 , -5.024132 , -4.5925217, ..., -1.2912979, -5.0251236,
        -4.335083 ],
       ...,
       [-2.1934204,  0.       , -1.445549 , ..., -2.4002   , -5.580246 ,
        -2.2692642],
       [-1.2422333,  0.3035125, -3.333397 , ..., -2.6573944, -4.5811615,
        -2.7085645],
       [ 0.       , -4.290077 , -3.11644  , ..., -3.1033401, -6.1597824,
        -5.277649 ]], shape=(89, 20), dtype=float32)

compare base vs fine tuned llr/dms spearman correlations

In [43]:
for key in keys:
    base_llr = dict_PF00030[key]['llr_pg2']
    lora_llr = dict_PF00030[key]['llr_pg2_lora_eps50_lr1eneg3_PF00030_CLM']
    dms_mat = dict_PF00030[key]['DMS_mat']

    # print(len(base_llr), len(lora_llr), len(dms_mat))


    spb = spearman_ignore_nan(base_llr, dms_mat)
    spl = spearman_ignore_nan(lora_llr, dms_mat)

    print(f"Base model spearman correlation with DMS is {spb[0]}")
    print(f"Lora model spearman correlation with DMS is {spl[0]}")
    print()

Base model spearman correlation with DMS is 0.4172571483248994
Lora model spearman correlation with DMS is 0.2105412739574139

Base model spearman correlation with DMS is 0.33897700677632076
Lora model spearman correlation with DMS is 0.1750848402774465

Base model spearman correlation with DMS is 0.36205722660833856
Lora model spearman correlation with DMS is 0.10390199985565027

Base model spearman correlation with DMS is 0.21845045525531706
Lora model spearman correlation with DMS is 0.08534220455322551

Base model spearman correlation with DMS is 0.3662332108928841
Lora model spearman correlation with DMS is 0.2213490234279677

Base model spearman correlation with DMS is 0.2790765656023974
Lora model spearman correlation with DMS is 0.08154157594993876

Base model spearman correlation with DMS is 0.43572135931376255
Lora model spearman correlation with DMS is 0.17499160761901747

Base model spearman correlation with DMS is 0.31101513899232713
Lora model spearman correlation with DM

try rescaling adapter

# Fine tune model using 30 epochs, learning rate of 1e-3, FeatureExtraction

In [38]:
lora_config = LoraConfig(
    r=8,
    lora_alpha=32,
    # target_modules=["query", "key", "value", "output.dense"],
    target_modules=["qkv_proj", "out_proj"],
    lora_dropout=0.1,
    bias="none",
    task_type=TaskType.FEATURE_EXTRACTION
    # task_type=TaskType.CAUSAL_LM
)

# protein_seq = 'EDGINLEEIREFAKNFKIRRLSLGLTQTQVGQALTATEGPAYSQSAICRFEKLDITPKSAQKLKPVLEKWLNEAELRNQEGQQNLMEFVG'
key = keys[0]
protein_seq = dict_PF00030[key]['wt_seq']

df_mutation = pd.read_csv('mutation.csv') # automate df_mutation
fitness_data = df_mutation
fitness_data.reset_index(drop=True, inplace=True)
positions = np.arange(len(protein_seq))

amino_acids = 'ACDEFGHIKLMNPQRSTVWY'
aa_token_ids = [tokenizer.convert_tokens_to_ids(aa) for aa in amino_acids]
positions_col = np.repeat(positions, len(amino_acids))
amino_acids_col = np.tile(list(amino_acids), len(positions))
df2 = pd.DataFrame({'real_position': positions_col, 'mut_aa': amino_acids_col})
df_merged = df2.merge(
    fitness_data,
    on=['real_position', 'mut_aa'],
    how='left')
fitness_list = df_merged['normalized_fitness'].tolist()
fitness_tensor = torch.tensor(fitness_list)
exp_tensor = fitness_tensor
seq_len = exp_tensor.shape[0]

loss = listwise_ranking_loss


# num_epochs = [30]
# l_rates = [1e-3, 5e-3]
# len_ep = len(num_epochs)
# len_lr = len(l_rates)

# results = {}


# for i in range(len_ep):
#     eps = num_epochs[i]
#     results[eps] = []
#     # print(f"Number of epochs: {eps}")
#     for j in range(len_lr):
#         lr = l_rates[j]
#         # print(f"Learning rate: {lr}")
eps = 30
lr = 1e-3
        
model, val_loss = FineTune_ProGen2_LORA(device, base_model, tokenizer, 
                                        lora_config, protein_seq, exp_tensor, 
                                        loss, lrate=lr, num_epochs=eps, k=0.5, 
                                        num_samples=20, print_info=False)

print(f"Validation loss = {val_loss}")
        

Validation loss = 2.1665737628936768


In [ ]:
# save a model
# model_dir = '/Users/johnhutchens/Desktop/Practicum/Models/'+'pg2_lora_eps30_lr1eneg3_PF00030'

# model.save_pretrained(model_dir)
# tokenizer.save_pretrained(model_dir)

('/Users/johnhutchens/Desktop/Practicum/Models/pg2_lora_eps30_lr1eneg3_PF00030/tokenizer_config.json',
 '/Users/johnhutchens/Desktop/Practicum/Models/pg2_lora_eps30_lr1eneg3_PF00030/special_tokens_map.json',
 '/Users/johnhutchens/Desktop/Practicum/Models/pg2_lora_eps30_lr1eneg3_PF00030/vocab.json',
 '/Users/johnhutchens/Desktop/Practicum/Models/pg2_lora_eps30_lr1eneg3_PF00030/merges.txt',
 '/Users/johnhutchens/Desktop/Practicum/Models/pg2_lora_eps30_lr1eneg3_PF00030/added_tokens.json',
 '/Users/johnhutchens/Desktop/Practicum/Models/pg2_lora_eps30_lr1eneg3_PF00030/tokenizer.json')

In [70]:
for key in dict_PF00030.keys():
    seq = dict_PF00030[key]['wt_seq']
    lp, rlp, llr = collect_log_prob_pg2(seq, model, tokenizer)
    dict_PF00030[key]['llr_pg2_lora_eps30_lr1eneg3_PF00030'] = llr

In [71]:
for key in dict_PF00030.keys():
    seq = dict_PF00030[key]['wt_seq']
    lp, rlp, llr = collect_log_prob_pg2(seq, base_model, tokenizer)
    dict_PF00030[key]['llr_pg2'] = llr

In [72]:
keys = list(dict_PF00030.keys())

In [73]:
i = 0

dict_PF00030[keys[i]]['llr_pg2'] - dict_PF00030[keys[i]]['llr_pg2_lora_eps30_lr1eneg3_PF00030']

array([[-2.59404   , -2.0913162 , -3.4876025 , ..., -2.593216  ,
         0.        , -0.77513885],
       [-3.553444  , -3.9058225 , -4.533386  , ..., -3.706253  ,
        -4.620201  , -4.8306885 ],
       [-2.7412338 , -4.2763214 , -5.300476  , ..., -1.0279007 ,
        -5.292282  , -4.0886993 ],
       ...,
       [-2.5057983 ,  0.        , -2.6578522 , ..., -2.8506927 ,
        -6.108902  , -2.8343887 ],
       [-1.4111786 ,  0.2569732 , -3.8256226 , ..., -2.8599472 ,
        -5.49057   , -3.2911375 ],
       [ 0.        , -4.3262253 , -3.6228716 , ..., -3.092659  ,
        -7.2896194 , -5.860054  ]], shape=(89, 20), dtype=float32)

In [ ]:
# path = '/Users/johnhutchens/Desktop/Practicum/Data/Domainome/'
# with open(path+"dict_PF00030.pkl", "wb") as f:
#     pickle.dump(dict_PF00030, f)

In [80]:
for key in keys:
    base_llr = dict_PF00030[key]['llr_pg2']
    lora_llr = dict_PF00030[key]['llr_pg2_lora_eps30_lr1eneg3_PF00030']
    dms_mat = dict_PF00030[key]['DMS_mat']

    # print(len(base_llr), len(lora_llr), len(dms_mat))


    spb = spearman_ignore_nan(base_llr, dms_mat)
    spl = spearman_ignore_nan(lora_llr, dms_mat)

    print(f"Base model spearman correlation with DMS is {spb[0]}")
    print(f"Lora model spearman correlation with DMS is {spl[0]}")
    print()

Base model spearman correlation with DMS is 0.4172571483248994
Lora model spearman correlation with DMS is 0.15932049492101447

Base model spearman correlation with DMS is 0.33897700677632076
Lora model spearman correlation with DMS is 0.17179156530206946

Base model spearman correlation with DMS is 0.36205722660833856
Lora model spearman correlation with DMS is 0.12497598169493365

Base model spearman correlation with DMS is 0.21845045525531706
Lora model spearman correlation with DMS is 0.16575524324784763

Base model spearman correlation with DMS is 0.3662332108928841
Lora model spearman correlation with DMS is 0.05460478145640364

Base model spearman correlation with DMS is 0.2790765656023974
Lora model spearman correlation with DMS is 0.07376780266484208

Base model spearman correlation with DMS is 0.43572135931376255
Lora model spearman correlation with DMS is 0.08960826853333739

Base model spearman correlation with DMS is 0.31101513899232713
Lora model spearman correlation with

In [50]:
dict_PF00030[keys[2]]['DMS_mat'] - dict_PF00030[keys[3]]['DMS_mat']

array([[0., 0., 0., ..., 0., 0., 0.],
       [0., 0., 0., ..., 0., 0., 0.],
       [0., 0., 0., ..., 0., 0., 0.],
       ...,
       [0., 0., 0., ..., 0., 0., 0.],
       [0., 0., 0., ..., 0., 0., 0.],
       [0., 0., 0., ..., 0., 0., 0.]], shape=(89, 20))